# Week 6 · 01 Gradient descent and backpropagation, slowly

You wrote that gradient descent doesn't click yet, that Week 5 · 03 §5 needed pictures of the matrices, and that §7 (vanishing gradients) wasn't clear. This notebook redoes those three things slowly. Nothing in it is new: gradient descent from one weight, then two, then a real model; then backpropagation on a batch with every matrix drawn and every number shown; then why gradients vanish. Two exercises, both from your booklet (1.8 and 13.7).

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **slope (derivative)**: how fast the loss changes when one weight changes: $dL/dw = -6$ means "raise $w$ by 0.01 and the loss drops by about 0.06".
- **gradient**: the list of slopes, one per weight. It points in the direction in which the loss rises fastest.
- **learning rate $\eta$**: the size of each step, as a fraction of the slope.
- **mask**: a table of 1s and 0s saying where a ReLU unit was on (1) or off (0).

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "01")      # defines check(n)

## M. The map: what's in this notebook and why

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and booklet 1.8 from memory | the gradient-descent loop you'll take apart in §3 |
| 1 | gradient descent with ONE weight | the whole idea on one curve, every number written out |
| 2 | two weights, then thousands | the gradient is just one slope per weight |
| 3 | a real model: booklet 1.8, line by line | the line `grad = 2 / n * X1.T @ (X1 @ w - y)` explained piece by piece |
| 4 | backprop on a batch, every matrix drawn | Week 5 · 03 §5 again, with pictures and numbers (exercise 2: booklet 13.7) |
| 5 | vanishing gradients, slowly | Week 5 · 03 §7 again, as a multiplication of many small numbers |

**How it connects.** Gradient descent needs one slope per weight. Backpropagation is the method that gets all of them at once. Vanishing gradients is what happens to those slopes in deep networks.

## 0. Recap first · 10 min, closed book

**Quick check:** What does the gradient descent update $w \leftarrow w - \eta\, \nabla L(w)$ do, in words?

<details><summary>Answer</summary>

It moves every weight a little in the direction that lowers the loss fastest; $\eta$ says how far. [Week 5 · 05 §1 ↗](../week5/05_optimizers.ipynb)

</details>

**Quick check:** $Z = XW + b$ for a batch. What is $\partial L / \partial W$ if $G = \partial L / \partial Z$?

<details><summary>Answer</summary>

$X^\top G$: one gradient for every weight, the same shape as $W$. [Week 5 · 01 §8 ↗](../week5/01_formula_to_numpy.ipynb)

</details>

**Quick check:** What does a ReLU do to the gradient flowing back through it?

<details><summary>Answer</summary>

Passes it unchanged where the unit was on ($z > 0$) and blocks it (0) where it was off.

</details>

**Quick check:** $\sigma'(z) = \sigma(z)(1 - \sigma(z))$. What is its largest value?

<details><summary>Answer</summary>

$0.25$, at $z = 0$ (where $\sigma = 0.5$, and $0.5 \times 0.5 = 0.25$).

</details>

**Quick check:** One epoch with 1000 samples and batch size 100: how many steps?

<details><summary>Answer</summary>

10: one step per batch. (`00` §B.)

</details>

**Exercise 1 · Booklet 1.8: linear regression by gradient descent.** `X1` is `(n, d+1)` with a first column of 1s (the intercept trick), `y` is `(n,)`. Start from $w = 0$; repeat `steps` times: $\nabla L = \frac{2}{n} X_1^\top (X_1 w - y)$, then $w \leftarrow w - \eta \nabla L$. Return `w`, **shape `(d+1,)`**.

**Example**: what goes in, and what should come out:
```python
X1 = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]]); y = np.array([1.0, 3.0, 5.0])    # y = 1 + 2x exactly
linreg_gd(X1, y, 0.1, 2000)    # → [1. 2.]   the intercept 1 and the slope 2
```

In [ ]:
def linreg_gd(X1, y, lr, steps):
    pass  # your code

In [ ]:
check(1)

## 1. Gradient descent with one weight

**The problem.** We want the weight with the smallest loss. If we could see the whole loss curve we'd just read off its lowest point, but we can't: for a network, each value of the loss costs a forward pass. What we **can** do at any $w$ is compute the loss there and its **slope** there (that's what the backward pass gives).

**The idea.** Think of standing on a hillside in fog. You can't see the valley, but you can feel which way the ground tilts under your feet. Take a small step downhill, feel again, and repeat. The slope tells you two things: which way is down (its sign), and roughly how far you still are from the bottom (its size).

**The formula, piece by piece.**

$$
w_{\text{new}} = w - \eta \cdot \frac{dL}{dw}
$$

- $\frac{dL}{dw}$ is the slope at the current $w$: how much the loss changes per unit change of $w$.
- If the slope is **negative** (the loss goes down to the right), $-\eta \cdot \text{slope}$ is positive, so $w$ moves **right**. If the slope is positive, $w$ moves left. Either way: downhill.
- $\eta$ (the learning rate) scales the step. A steep slope gives a big step; near the bottom, where the slope is almost 0, the steps get small by themselves.

**An example.** $L(w) = (w - 3)^2 + 1$. Its lowest point is at $w = 3$ (we know, but the algorithm doesn't). The slope is $\frac{dL}{dw} = 2(w - 3)$. Start at $w = 0$ with $\eta = 0.1$:

🖼 **Picture.** Left: the first step. At $w = 0$ the slope is $-6$, so the step is $-0.1 \times (-6) = +0.6$. Right: eight steps; they shrink as the curve flattens.

In [ ]:
fig_gd_1d()

| step $t$ | $w$ | slope $2(w - 3)$ | step $-0.1 \times$ slope | new $w$ | distance to 3 |
|---|---|---|---|---|---|
| 0 | 0 | −6 | +0.6 | 0.6 | 3 → 2.4 |
| 1 | 0.6 | −4.8 | +0.48 | 1.08 | 2.4 → 1.92 |
| 2 | 1.08 | −3.84 | +0.384 | 1.464 | 1.92 → 1.536 |
| 3 | 1.464 | −3.072 | +0.307 | 1.771 | 1.536 → 1.229 |

Look at the last column: every step removes 20% of the remaining distance. That's because here the slope is proportional to the distance: $2(w - 3) = -2 \times \text{distance}$, so the step is $0.1 \times 2 = 0.2$ times the distance. Gradient descent never jumps to the answer; it creeps towards it.

In [ ]:
w = 0.0
for t in range(4):
    slope = 2 * (w - 3)                 # dL/dw at the current w
    w = w - 0.1 * slope                 # one step downhill
    print(t, round(w, 3))
# → 0 0.6
#   1 1.08
#   2 1.464
#   3 1.771

**In one sentence:** gradient descent repeatedly moves the weight a little in the direction the slope says is downhill.

## 2. Two weights, then thousands

**The problem.** A real model has many weights. Each one has its own slope: how the loss changes if **that** weight moves and all the others stay put. That's a **partial derivative**, written $\partial L / \partial w_1$.

**The idea.** Put the slopes in a list: that list is the **gradient**, $\nabla L = (\partial L / \partial w_1, \partial L / \partial w_2, \dots)$. Then update every weight with its own slope, all at the same time:

$$
w_1 \leftarrow w_1 - \eta\, \frac{\partial L}{\partial w_1}, \qquad w_2 \leftarrow w_2 - \eta\, \frac{\partial L}{\partial w_2}, \qquad \text{i.e.} \quad w \leftarrow w - \eta\, \nabla L
$$

With two weights the loss is a landscape over the $(w_1, w_2)$ plane, drawn with contour lines (like a map). The gradient is an arrow pointing **uphill**, the steepest way up; the step goes the opposite way.

🖼 **Picture.** $L = \frac{1}{2}(w_1^2 + 4 w_2^2)$, starting at $(-3, 1.5)$ with $\eta = 0.2$.

In [ ]:
fig_gd_2d()

**By hand, the first step.** $\partial L / \partial w_1 = w_1 = -3$ and $\partial L / \partial w_2 = 4 w_2 = 6$, so $\nabla L = (-3, 6)$. The step is $-0.2 \times (-3, 6) = (0.6, -1.2)$, giving $(-2.4, 0.3)$. The $w_2$ direction is steeper (its slope is 6), so it moves further.

In [ ]:
w = np.array([-3.0, 1.5])
grad = np.array([1.0, 4.0]) * w          # (dL/dw1, dL/dw2) = (w1, 4 w2)
print(grad, w - 0.2 * grad)               # → [-3.  6.] [-2.4  0.3]

**Thousands of weights.** The digits network of Week 5 · 06 has 2410 weights, so its gradient is a list of 2410 slopes, and the update is the same line: every weight minus $\eta$ times its own slope. The only hard part is getting 2410 slopes cheaply; that's backpropagation (§4).

**In one sentence:** the gradient is one slope per weight, and gradient descent steps every weight against its own slope at once.

## 3. A real model: booklet 1.8, line by line

**Recap (your booklet, 1.8).** `w = np.zeros(d + 1)`, then 500 times: `grad = 2 / n * X1.T @ (X1 @ w - y)` and `w = w - 0.1 * grad`. It lands on the closed-form answer.

**The line that may look odd:** `grad = 2 / n * X1.T @ (X1 @ w - y)`. It's the gradient of the mean squared error $L(w) = \frac{1}{n}\sum_i (x_i^\top w - y_i)^2$, computed for all $d + 1$ weights at once. Piece by piece:

| Piece | Shape | What it is |
|---|---|---|
| `X1 @ w` | `(n,)` | every sample's prediction $\hat y_i = x_i^\top w$ |
| `X1 @ w - y` | `(n,)` | every sample's error $e_i = \hat y_i - y_i$ |
| `X1.T @ (...)` | `(d+1,)` | for each weight $j$: $\sum_i x_{ij}\, e_i$, the errors weighted by that weight's feature |
| `2 / n * ...` | `(d+1,)` | the derivative of the mean of squares: $\frac{\partial L}{\partial w_j} = \frac{2}{n}\sum_i e_i\, x_{ij}$ |

**Why $x_{ij}\, e_i$?** Weight $j$ affects sample $i$'s prediction by $x_{ij}$ per unit (if $x_{ij} = 0$, moving $w_j$ does nothing for that sample). The error $e_i$ says how much that sample wants its prediction to change. Their product, summed over samples, is how much the loss wants $w_j$ to change.

In [ ]:
X1 = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]]); y = np.array([1.0, 3.0, 5.0])   # 3 samples: y = 1 + 2x
w = np.zeros(2)
e = X1 @ w - y                       # errors at w = 0: every prediction is 0
print(e, (2 / 3 * X1.T @ e).round(3))    # → [-1. -3. -5.] [-6.    -8.667]

Read the gradient: $\partial L / \partial w_0 = \frac{2}{3}(-1 - 3 - 5) = -6$ (all errors push the intercept up), and $\partial L / \partial w_1 = \frac{2}{3}(0 \cdot (-1) + 1 \cdot (-3) + 2 \cdot (-5)) = -8.667$ (samples with bigger $x$ push the slope harder). Both are negative, so both weights go up.

**With minibatches** (Week 5 · 05 §2) it's the same line, with `X1` and `y` replaced by one batch's rows and $n$ by the batch size. The gradient is then an estimate, right on average.

## 4. Backpropagation on a batch, every matrix drawn

**The problem.** A network has layers, so the loss depends on the first layer's weights only through everything after them. The slope for $W_1$ isn't one line like in §3. Backprop computes it by going backwards through the layers, reusing what it computed for the layer above.

**The setup.** A batch of 4 samples with 3 features each; a hidden layer of 5 ReLU units; 2 classes with softmax and cross-entropy. Each picture box below is a matrix, drawn with its real shape.

🖼 **Picture.** Top: the forward pass keeps every matrix. Bottom: the backward pass. Each step uses the step before it (to its left). Every gradient comes out with the shape of the thing it belongs to.

In [ ]:
fig_backprop_shapes()

**The forward pass, with numbers**

In [ ]:
X = np.array([[1, 0, 2], [0, 1, 1], [2, 1, 0], [1, 1, 1.0]])            # (4, 3): 4 samples, 3 features
W1 = np.array([[1, -1, 0, 1, 0], [0, 1, -1, 1, 1], [1, 0, 1, -1, 1.0]])  # (3, 5): 3 inputs -> 5 hidden units
W2 = np.array([[1, 0], [0, 1], [1, -1], [-1, 1], [0, 1.0]])            # (5, 2): 5 hidden units -> 2 classes
y = np.array([0, 1, 1, 0]); Y = np.eye(2)[y]                            # true classes, and one-hot (4, 2)

Z1 = X @ W1                          # (4, 5)  (biases are 0 here, to keep the numbers small)
H1 = np.maximum(Z1, 0)               # (4, 5)  ReLU
Z2 = H1 @ W2                         # (4, 2)  logits
P = np.exp(Z2) / np.exp(Z2).sum(axis=1, keepdims=True)    # (4, 2)  softmax, row by row
print(P.round(3))
# → [[0.993 0.007]
#    [0.119 0.881]
#    [0.007 0.993]
#    [0.119 0.881]]

Compare with `y = [0, 1, 1, 0]`: samples 0, 1 and 2 are predicted well. Sample 3's true class is 0, but the network gives class 0 only 0.119. Watch how almost all of the gradient comes from that one sample.

**The backward pass, one step at a time**

**Step 1, the output gradient:** $G_2 = \frac{1}{n}(P - Y)$, the $p - y$ rule divided by the batch size $n = 4$ (the loss is a mean over the batch). Shape `(4, 2)`: one row per sample.

In [ ]:
G2 = (P - Y) / 4                     # (4, 2)
print(G2.round(3))
# → [[-0.002  0.002]
#    [ 0.03  -0.03 ]
#    [ 0.002 -0.002]
#    [-0.22   0.22 ]]

Row 3 is the large one, $(-0.22, 0.22)$: "raise class 0's score, lower class 1's" for sample 3. The well-predicted samples contribute almost nothing.

**Step 2, the last layer's weights:** $dW_2 = H_1^\top G_2$. Shape: $(5, 4) @ (4, 2) = (5, 2)$, the shape of $W_2$. Entry $(k, c)$ adds up, over the 4 samples, (how active hidden unit $k$ was) × (how much class $c$'s score should change). By hand for entry $(0, 0)$: hidden unit 0's values are $H_1[:, 0] = (3, 1, 2, 2)$, and class 0's gradients are $G_2[:, 0] = (-0.002, 0.030, 0.002, -0.220)$:

$$
dW_2[0, 0] = 3(-0.002) + 1(0.030) + 2(0.002) + 2(-0.220) = -0.005 + 0.030 + 0.003 - 0.440 = -0.412
$$

In [ ]:
dW2 = H1.T @ G2                      # (5, 4) @ (4, 2) -> (5, 2)
print(dW2.round(3))
# → [[-0.412  0.412]
#    [ 0.03  -0.03 ]
#    [-0.003  0.003]
#    [-0.215  0.215]
#    [-0.382  0.382]]

**Step 3, pass the gradient back to the hidden layer:** $G_2 W_2^\top$, shape $(4, 2) @ (2, 5) = (4, 5)$: for each sample, how much the loss wants each hidden unit's value to change. Then the **mask**: a ReLU unit that was off ($Z_1 \le 0$) passes nothing back, so multiply by 1 where $Z_1 > 0$ and 0 elsewhere. That gives $G_1$, shape `(4, 5)`.

In [ ]:
mask = (Z1 > 0).astype(float)        # (4, 5): 1 = the unit was on
G1 = np.where(Z1 > 0, G2 @ W2.T, 0.0) # (4, 5): pass back, then block where the unit was off
print(mask)
print(G1.round(3))
# → [[1. 0. 1. 0. 1.]
#    [1. 1. 0. 0. 1.]
#    [1. 0. 0. 1. 1.]
#    [1. 0. 0. 1. 1.]]
#   [[-0.002  0.    -0.003  0.     0.002]
#    [ 0.03  -0.03   0.     0.    -0.03 ]
#    [ 0.002  0.     0.    -0.003 -0.002]
#    [-0.22   0.     0.     0.44   0.22 ]]

(`np.where(condition, a, b)` takes `a` where the condition is True and `b` elsewhere. It equals `(G2 @ W2.T) * (Z1 > 0)`, the form in your booklet.)

**Step 4, the first layer's weights:** $dW_1 = X^\top G_1$, shape $(3, 4) @ (4, 5) = (3, 5)$, the shape of $W_1$. The same rule as step 2, one layer further back: (input value) × (gradient at the unit), summed over the batch.

In [ ]:
dW1 = X.T @ G1                       # (3, 4) @ (4, 5) -> (3, 5)
print(dW1.round(3))
# → [[-0.219  0.    -0.003  0.434  0.219]
#    [-0.189 -0.03   0.     0.437  0.189]
#    [-0.194 -0.03  -0.007  0.44   0.194]]

**Check against PyTorch's autograd**, which knows nothing about our four steps:

In [ ]:
import torch
t1, t2 = torch.tensor(W1, requires_grad=True), torch.tensor(W2, requires_grad=True)
logits = torch.relu(torch.tensor(X) @ t1) @ t2
torch.nn.functional.cross_entropy(logits, torch.tensor(y)).backward()     # mean cross-entropy, then backprop
print(np.allclose(t1.grad.numpy(), dW1), np.allclose(t2.grad.numpy(), dW2))   # → True True

**The pattern, layer by layer** (your booklet's algorithm card, 13.5): **weights' gradient = (this layer's input)ᵀ @ G**; **bias gradient = column sums of G**; **pass back = G @ Wᵀ, then the mask**. For a network with more layers, the last two lines repeat once per layer.

Your booklet's 13.7 code keeps the forward pass's matrices in two lists, `Hs` (each layer's input; `Hs[0]` is `X`) and `Zs` (each layer's sum before ReLU). Here is its `forward`; exercise 2 is its `backward`.

In [ ]:
from scipy.special import softmax

def forward(X, Ws, bs):                               # booklet 13.7, step 1
    Hs, Zs = [X], []
    for l, (W, b) in enumerate(zip(Ws, bs)):
        Zs.append(Hs[-1] @ W + b)                     # Z_l = H_{l-1} W_l + b_l
        Hs.append(np.maximum(Zs[-1], 0) if l < len(Ws) - 1 else Zs[-1])   # no ReLU on the logits
    return Hs, Zs

Hs, Zs = forward(X, [W1, W2], [np.zeros(5), np.zeros(2)])
print([h.shape for h in Hs], [z.shape for z in Zs])   # → [(4, 3), (4, 5), (4, 2)] [(4, 5), (4, 2)]

**Exercise 2 · Booklet 13.7: the backward function.** Given `Hs`, `Zs` from `forward`, the one-hot labels `Y` `(n, C)` and the weights `Ws`, return the list of `(dW, db)` pairs, **first layer first**. Start with $G = (\operatorname{softmax}(Z_L) - Y)/n$ (`Hs[-1]` holds the logits); for $l$ from the last layer down: $dW_l = $ `Hs[l].T @ G`, $db_l = $ `G.sum(axis=0)`, and, if $l > 0$, $G \leftarrow$ `(G @ Ws[l].T) * (Zs[l - 1] > 0)`.

**Example**: what goes in, and what should come out:
```python
grads = backward(Hs, Zs, Y, [W1, W2])     # the batch above
grads[1][0].round(3)     # → dW2 (5, 2), as in step 2: [[-0.412  0.412] ...
grads[0][0].shape        # → (3, 5)
```

In [ ]:
def backward(Hs, Zs, Y, Ws):
    pass  # your code

In [ ]:
check(2)

## 5. Vanishing gradients, slowly

**The problem.** Backprop goes from the loss back towards the input, and at every layer the gradient is **multiplied** by that layer's local gradient (`00` §B). Multiply many numbers smaller than 1 and you get a tiny number. With 10 sigmoid layers the gradient reaching the first layer can be a million times smaller than at the last: the first layers barely change, so the network barely learns.

**Where the small numbers come from.** At a sigmoid unit the local gradient is $\sigma'(z) = \sigma(z)(1 - \sigma(z))$. Its largest possible value is $0.5 \times 0.5 = 0.25$, at $z = 0$; for large $|z|$ it's close to 0 (the curve is flat there). So each sigmoid layer multiplies the gradient by **at most 0.25** (times the weights).

**A tiny example by hand.** A chain of 4 sigmoid units, one after another, every weight 1, every unit at $z = 0$ (the best case, $\sigma' = 0.25$). The gradient of the output with respect to the input is the product of the 4 local gradients:

| Going back through | multiply by | gradient so far |
|---|---|---|
| unit 4 (last) | 0.25 | 0.25 |
| unit 3 | 0.25 | 0.0625 |
| unit 2 | 0.25 | 0.0156 |
| unit 1 (first) | 0.25 | 0.0039 |

Even in the best case, 4 layers shrink the gradient 256 times ($4^4$); 10 layers, about a million times ($4^{10}$). With ReLU, the local gradient of an active unit is exactly 1, so nothing shrinks just from the activation.

🖼 **Picture.** The largest possible gradient after each layer, going back from the loss: sigmoid (×0.25 per layer at best) against ReLU (×1 for active units).

In [ ]:
fig_vanishing_chain()

**Measured in a real 10-layer network** (32 units per layer, a batch of 64): the size of $\partial L / \partial W$ in each layer, first layer first. With sigmoids it grows from about $10^{-9}$ at the first layer to $10^{-2}$ at the last. With ReLU and He initialization (`03` §1) it stays between about 0.3 and 0.8:

In [ ]:
torch.manual_seed(0)
for name, act, he in [("sigmoid", torch.sigmoid, False), ("ReLU + He", torch.relu, True)]:
    torch.manual_seed(0)
    layers = [torch.nn.Linear(32, 32) for _ in range(10)]
    if he:
        for lin in layers:
            torch.nn.init.kaiming_normal_(lin.weight, nonlinearity="relu"); torch.nn.init.zeros_(lin.bias)
    out = torch.nn.Linear(32, 10)
    h = torch.randn(64, 32)
    for lin in layers:
        h = act(lin(h))
    torch.nn.functional.cross_entropy(out(h), torch.randint(0, 10, (64,))).backward()
    print(name, [f"{lin.weight.grad.norm().item():.0e}" for lin in layers])
# → sigmoid ['2e-09', '8e-09', '5e-08', '3e-07', '2e-06', '2e-05', '1e-04', '8e-04', '6e-03', '4e-02']
#   ReLU + He ['3e-01', '3e-01', '5e-01', '4e-01', '7e-01', '6e-01', '7e-01', '6e-01', '8e-01', '6e-01']

**The fixes**, each in `03` this week: ReLU activations; initialization scaled to the layer size (He), so signals neither shrink nor grow; batch normalization, which re-centres each layer's values; and residual connections, which add a path where the gradient is multiplied by 1.

**In one sentence:** backprop multiplies one factor per layer, sigmoid factors are at most 0.25, so in deep sigmoid networks the gradient fades to almost nothing before it reaches the first layers.

In [ ]:
for n in range(1, 3):
    check(n)

---
Solutions: `solutions/01_solutions.py`. Next: `02_regularization.ipynb`.